# Gender Classification Benchmark

Compares all classification methods against GND as ground truth.

**Methods covered:** gender_guesser (gg), HF gender classification (hfgc), HF genderize (hfg), HIVOTO (hv), NQG (nqg), LLM (llm — may be partial)

**Ground truth:** GND records where `gnd_gender_mapped` is `male` or `female` (excludes `no_gnd` and `unknown`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

RESULTS = Path("../data/results")

# ── load all method results ──────────────────────────────────────────────────
gnd  = pd.read_csv(RESULTS / "gnd_gender_results.csv", dtype=str)
gg   = pd.read_csv(RESULTS / "gender_guesser_results.csv", dtype=str)
hfgc = pd.read_csv(RESULTS / "hf_gender_classification_results.csv", dtype=str)
hfg  = pd.read_csv(RESULTS / "hf_genderize_results.csv", dtype=str)
hv   = pd.read_csv(RESULTS / "hivoto_results.csv", dtype=str)
nqg  = pd.read_csv(RESULTS / "nqg_results.csv", dtype=str)
llm  = pd.read_csv(RESULTS / "llm_gender_results.csv", dtype=str)

print(f"GND    : {len(gnd):6,} rows")
print(f"gg     : {len(gg):6,} rows")
print(f"hfgc   : {len(hfgc):6,} rows")
print(f"hfg    : {len(hfg):6,} rows")
print(f"hv     : {len(hv):6,} rows")
print(f"nqg    : {len(nqg):6,} rows")
print(f"llm    : {len(llm):6,} rows  ← still running")

In [ ]:
# ── select only the columns we need from each method ────────────────────────
gg_s   = gg  [["xml_id", "gg_gender"]]
hfgc_s = hfgc[["xml_id", "hfgc_gender"]]
hfg_s  = hfg [["xml_id", "hfg_gender"]]
hv_s   = hv  [["xml_id", "hv_gender"]]
nqg_s  = nqg [["xml_id", "nqg_gender"]]
llm_s  = llm [["xml_id", "llm_gender"]]
gnd_s  = gnd [["xml_id", "gnd_gender_mapped"]]

# ── merge all into one wide DataFrame ───────────────────────────────────────
base = gnd_s.copy()
for df in [gg_s, hfgc_s, hfg_s, hv_s, nqg_s, llm_s]:
    base = base.merge(df, on="xml_id", how="outer")

# method columns and friendly names
METHODS = {
    "gg_gender"   : "gender_guesser",
    "hfgc_gender" : "HF ClassifyGender",
    "hfg_gender"  : "HF Genderize",
    "hv_gender"   : "HIVOTO",
    "nqg_gender"  : "NQG",
    "llm_gender"  : "LLM (partial)",
}

print(f"Combined DataFrame: {len(base):,} rows × {base.shape[1]} columns")
base.head(5)

## 1 · Ground Truth from GND

Records where GND returned a definite gender (`male` or `female`).  
`unknown` and `no_gnd` entries are excluded from precision evaluation.

In [ ]:
gt = gnd[gnd["gnd_gender_mapped"].isin(["male", "female"])].copy()

print(f"Total GND records        : {len(gnd):>6,}")
print(f"  no GND match           : {(gnd['gnd_gender_mapped']=='no_gnd').sum():>6,}")
print(f"  GND gender = unknown   : {(gnd['gnd_gender_mapped']=='unknown').sum():>6,}")
print(f"  GND gender = male      : {(gnd['gnd_gender_mapped']=='male').sum():>6,}")
print(f"  GND gender = female    : {(gnd['gnd_gender_mapped']=='female').sum():>6,}")
print(f"\nGround truth subset size : {len(gt):>6,}  (male + female only)")

## 2 · Coverage Statistics per Method

- **classified**: result is `male` or `female`  
- **ambiguous**: result is `conflict`, `unknown`, `uncertain`, `andy`, or any variant  
- **failed**: `no_name` or NaN (no name was parseable)

In [ ]:
CLASSIFIED  = {"male", "female"}
AMBIGUOUS   = {"conflict", "unknown", "uncertain", "andy", "mostly_male", "mostly_female"}
FAILED      = {"no_name"}

def categorise(val):
    if pd.isna(val) or val == "":
        return "failed"
    v = str(val).strip().lower()
    if v in CLASSIFIED:
        return "classified"
    if v in FAILED:
        return "failed"
    return "ambiguous"

rows = []
for col, name in METHODS.items():
    # Use the appropriate source DataFrame (some methods may have more rows after merge)
    series = base[col] if col in base.columns else pd.Series(dtype=str)
    total = len(series)
    cats  = series.apply(categorise)
    n_classified = (cats == "classified").sum()
    n_ambiguous  = (cats == "ambiguous").sum()
    n_failed     = (cats == "failed").sum()
    rows.append({
        "Method"         : name,
        "Total"          : total,
        "Classified"     : n_classified,
        "Ambiguous"      : n_ambiguous,
        "Failed/no_name" : n_failed,
        "Coverage %"     : round(n_classified / total * 100, 1),
    })

coverage_df = pd.DataFrame(rows).set_index("Method")
coverage_df

## 3 · Classification Distribution per Method

Breakdown of exact label values (male / female / conflict / unknown / no_name / NaN).

In [ ]:
LABEL_ORDER = ["male", "female", "conflict", "unknown", "uncertain",
               "andy", "mostly_male", "mostly_female", "no_name", "NaN/missing"]

dist_rows = []
for col, name in METHODS.items():
    series = base[col].copy() if col in base.columns else pd.Series(dtype=str)
    # normalise NaN / empty
    series = series.fillna("NaN/missing").str.strip().str.lower()
    series = series.replace("", "NaN/missing")
    counts = series.value_counts()
    total  = len(series)
    row = {"Method": name}
    for lbl in LABEL_ORDER:
        row[lbl] = counts.get(lbl, 0)
        row[lbl + " %"] = round(counts.get(lbl, 0) / total * 100, 1)
    dist_rows.append(row)

dist_df = pd.DataFrame(dist_rows).set_index("Method")
# show just counts for readability
count_cols = [l for l in LABEL_ORDER]
dist_df[count_cols]

## 4 · Precision Evaluation Against GND Ground Truth

We restrict to the subset of records where GND has a definite gender (`male` or `female`).

For each method on this subset we compute:
- **n_gt_overlap**: how many of those GT records this method also processed
- **n_match**: method result equals GND result
- **n_conflict/unknown**: method returned an ambiguous answer (counts as wrong)
- **n_failed**: method returned no_name / NaN (not counted in accuracy denominator)
- **Accuracy**: matches / (overlap − failed)  — i.e. among records where the method gave *any* gender answer
- **Coverage on GT**: classified (male or female) / total GT overlap

In [ ]:
prec_rows = []

for col, name in METHODS.items():
    # merge ground truth with this method's predictions
    method_df = base[["xml_id", col]].copy()
    merged = gt[["xml_id", "gnd_gender_mapped"]].merge(method_df, on="xml_id", how="left")

    total_gt      = len(merged)
    pred          = merged[col].fillna("NaN/missing").str.strip().str.lower()
    pred          = pred.replace("", "NaN/missing")

    n_failed      = pred.isin({"no_name", "nan/missing"}).sum()
    n_classified  = pred.isin({"male", "female"}).sum()
    n_ambiguous   = total_gt - n_failed - n_classified

    # accuracy: among records the method actually classified (male/female)
    answered_mask = pred.isin({"male", "female"})
    n_match       = (pred[answered_mask] == merged.loc[answered_mask, "gnd_gender_mapped"]).sum()
    accuracy      = round(n_match / n_classified * 100, 1) if n_classified > 0 else None

    # per-class
    male_gt   = merged["gnd_gender_mapped"] == "male"
    female_gt = merged["gnd_gender_mapped"] == "female"
    n_match_m = ((pred == "male")   & male_gt).sum()
    n_match_f = ((pred == "female") & female_gt).sum()

    prec_rows.append({
        "Method"            : name,
        "GT records"        : total_gt,
        "Classified"        : n_classified,
        "Ambiguous"         : n_ambiguous,
        "Failed/no_name"    : n_failed,
        "Correct"           : int(n_match),
        "Accuracy %"        : accuracy,
        "Correct male"      : int(n_match_m),
        "Correct female"    : int(n_match_f),
    })

prec_df = pd.DataFrame(prec_rows).set_index("Method")
prec_df

## 5 · Summary Table

One row per method: overall coverage + precision on the GND ground-truth subset.

In [ ]:
summary = coverage_df[["Classified", "Ambiguous", "Failed/no_name", "Coverage %"]].copy()
summary["GT records"] = prec_df["GT records"]
summary["Correct on GT"] = prec_df["Correct"]
summary["Accuracy on GT %"] = prec_df["Accuracy %"]

# highlight best value per numeric column
def highlight_max(s):
    is_max = s == s.max()
    return ["background-color: #c6efce; font-weight: bold" if v else "" for v in is_max]

def highlight_min(s):
    is_min = s == s.min()
    return ["background-color: #ffc7ce" if v else "" for v in is_min]

(
    summary
    .style
    .apply(highlight_max, subset=["Coverage %", "Accuracy on GT %"])
    .apply(highlight_min, subset=["Coverage %", "Accuracy on GT %"])
    .format({"Coverage %": "{:.1f}%", "Accuracy on GT %": lambda v: f"{v:.1f}%" if v is not None else "N/A"})
)

## 6 · Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
methods = list(METHODS.values())

# ── Coverage % ───────────────────────────────────────────────────────────────
ax = axes[0]
cov_vals = [coverage_df.loc[m, "Coverage %"] for m in methods]
bars = ax.barh(methods, cov_vals, color=sns.color_palette("Blues_d", len(methods)))
ax.set_xlabel("Coverage %")
ax.set_title("Coverage: % of records classified\n(male or female)")
ax.set_xlim(0, 105)
for bar, val in zip(bars, cov_vals):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)

# ── Accuracy on GT % ─────────────────────────────────────────────────────────
ax = axes[1]
acc_vals = [prec_df.loc[m, "Accuracy %"] if prec_df.loc[m, "Accuracy %"] is not None else 0
            for m in methods]
bars = ax.barh(methods, acc_vals, color=sns.color_palette("Greens_d", len(methods)))
ax.set_xlabel("Accuracy %")
ax.set_title("Accuracy on GND ground truth\n(among classified records)")
ax.set_xlim(0, 105)
for bar, val in zip(bars, acc_vals):
    label = f"{val:.1f}%" if val > 0 else "N/A (partial)"
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            label, va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Stacked bar: classification distribution per method ──────────────────────
plot_labels = ["male", "female", "conflict", "unknown", "no_name", "NaN/missing"]
colors = ["#4472C4", "#ED7D31", "#FFC000", "#A9D18E", "#BFBFBF", "#E2EFDA"]

fig, ax = plt.subplots(figsize=(14, 5))
totals = {m: sum(dist_df.loc[m, l] for l in plot_labels) for m in methods}

left = np.zeros(len(methods))
for lbl, col in zip(plot_labels, colors):
    vals = np.array([
        dist_df.loc[m, lbl] / totals[m] * 100 if totals[m] > 0 else 0
        for m in methods
    ])
    ax.barh(methods, vals, left=left, label=lbl, color=col)
    # label segments wider than 3 %
    for i, (v, l) in enumerate(zip(vals, left)):
        if v > 3:
            ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=8)
    left += vals

ax.set_xlabel("Share of total records (%)")
ax.set_title("Classification distribution per method")
ax.set_xlim(0, 100)
ax.legend(loc="lower right", ncol=3)
plt.tight_layout()
plt.show()

## 7 · Misclassifications on GND Ground Truth

For each method: how many of the GND-classified records were wrong, and in which direction?

In [ ]:
mis_rows = []
for col, name in METHODS.items():
    merged = gt[["xml_id", "gnd_gender_mapped"]].merge(
        base[["xml_id", col]], on="xml_id", how="left"
    )
    pred = merged[col].fillna("NaN/missing").str.strip().str.lower().replace("", "NaN/missing")
    answered = pred.isin({"male", "female"})
    wrong = answered & (pred != merged["gnd_gender_mapped"])

    # direction of error
    # male predicted as female
    m_as_f = ((pred == "female") & (merged["gnd_gender_mapped"] == "male")).sum()
    # female predicted as male
    f_as_m = ((pred == "male")   & (merged["gnd_gender_mapped"] == "female")).sum()

    mis_rows.append({
        "Method"          : name,
        "Answered"        : int(answered.sum()),
        "Wrong total"     : int(wrong.sum()),
        "Error %"         : round(wrong.sum() / answered.sum() * 100, 1) if answered.sum() > 0 else None,
        "Male→Female"     : int(m_as_f),
        "Female→Male"     : int(f_as_m),
    })

pd.DataFrame(mis_rows).set_index("Method")